# 14차시 합성곱 신경망(CNN)으로 옷 맞히기
지난 시간에는 손글씨 숫자를 한 줄로 펼쳐서(`Flatten`) 맞혔습니다. 오늘은 **그림의 모양을 보면서** 옷 사진을 읽는 모델을 만듭니다. 티셔츠, 바지, 운동화... AI가 몇 가지나 구별할까요?

**오늘 새로 배우는 것 4가지**
1. **필터**: 이미지 위를 작은 창(3×3)이 훑으며 선·모서리 같은 모양을 찾는 도구
2. **합성곱 층(`Conv2D`)**: 필터를 여러 개 쓰는 층
3. **풀링(`MaxPooling2D`)**: 이미지 크기를 절반으로 줄이는 층
4. **채널**: 이미지 한 장의 모양 `(높이, 너비, 채널)`. 흑백은 채널이 1

위에서부터 순서대로 한 셀씩 실행하세요. 원본 코드 아래 **🔁 변형** 셀은 조건이 바뀌었을 때 숫자만 바꿔 쓰는 용도입니다. (변형 모델은 이름을 따로 붙여서 `model`, `history`를 덮어쓰지 않습니다.)

**목차**: A. 공식 정리 / 1. 데이터 준비 / 1-1. 필터 체험 / 1-2. 채널 / 2. CNN 모델 / 3. 학습 / 4. 학습 곡선 / 5. 13차시 모델과 비교 / 6. AI의 답 / 7. 개념 문제 / 8. 논술형
> ⚠ 표시 변형은 수업에 직접 나오지 않은 대비용 (컬러 사진)

# A. 공식 정리
### 출력 크기
| 층 | 들어가는 모양 | 나오는 모양 | 규칙 |
|---|---|---|---|
| `Input(shape=(28, 28, 1))` | - | `(28, 28, 1)` | (높이, 너비, 채널). 흑백 1, 컬러 3 |
| `Conv2D(32, (3, 3))` | `(28, 28, 1)` | `(26, 26, 32)` | 가로·세로 = **28 - 3 + 1 = 26**, 마지막 = **필터 수** |
| `Conv2D(32, (5, 5))` | `(28, 28, 1)` | `(24, 24, 32)` | 28 - 5 + 1 = 24 |
| `MaxPooling2D((2, 2))` | `(26, 26, 32)` | `(13, 13, 32)` | 가로·세로 **절반** (홀수면 버림: 11 → 5), 채널 그대로 |
| `Flatten()` | `(13, 13, 32)` | `(5408,)` | **모두 곱하기**: 13 × 13 × 32 |
| `Dense(10)` | `(5408,)` | `(10,)` | 노드 수 |

### 파라미터 계산
| 층 | 공식 | 예시 |
|---|---|---|
| `Conv2D` | **(필터 높이 × 필터 너비 × 들어오는 채널 + 1) × 필터 수** | (3×3×1 + 1) × 32 = **320** |
| `Dense` | **(들어오는 수 + 1) × 노드 수** | (5408 + 1) × 10 = **54,090** |
| `MaxPooling2D`, `Flatten`, `Input` | 학습할 값 없음 | **0** |

### 미리 계산해 둔 변형 모델 (직접 확인한 값)
| 모델 | 모양 흐름 | 합계 |
|---|---|---|
| 수업 CNN | (26,26,32) → (13,13,32) → 5408 → 10 | **54,410** |
| 13차시 Dense (채널 1 입력) | 784 → 128 → 10 | **101,770** |
| 필터 16개 | (26,26,16) → (13,13,16) → 2704 → 10 | 27,210 |
| 필터 64개 | (26,26,64) → (13,13,64) → 10816 → 10 | 108,810 |
| 필터 5x5 | (24,24,32) → (12,12,32) → 4608 → 10 | 46,922 |
| Conv 2층 (32 → 64) | ... → (11,11,64) → (5,5,64) → 1600 → 10 | 34,826 |
| CNN + Dense(64) | ... → 5408 → 64 → 10 | 347,146 |
| 풀링 없이 | (26,26,32) → 21632 → 10 | 216,650 |
| 출력 2종류 | ... → 5408 → 2 | 11,138 |
| ⚠ 컬러 28x28x3 | Conv2D (3×3×3+1)×32 = 896 | 54,986 |
| ⚠ 컬러 32x32x3 | (30,30,32) → (15,15,32) → 7200 → 10 | 72,906 |

# 1. 데이터 준비

In [ ]:
# 라이브러리 불러오기
import numpy as np                                                              # 숫자표(배열) 만들고 계산하는 도구
import matplotlib.pyplot as plt                                                 # 사진과 그래프 그리는 도구
import os                                                                       # 파일이 있는지 확인하는 도구
import cv2                                                                      # 사진 읽기·크기 바꾸기·필터 적용 도구
from skimage import data                                                        # 예제 사진(고양이)
from tensorflow.keras.datasets import fashion_mnist                             # 옷 사진 데이터
from tensorflow.keras.models import Sequential                                  # 층을 쌓아 모델을 만드는 틀
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense # 모델에 쌓을 층들

In [ ]:
# 옷 사진 데이터 불러오기 + 0~1로 만들기
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()    # (학습용 사진, 정답), (테스트용 사진, 정답)
X_train = X_train / 255                                             # 0~255 -> 0~1 (가장 큰 값 255로 나누기)
X_test = X_test / 255                                               # 테스트용도 똑같이

# 숫자 0~9에 해당하는 옷 이름
names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',         # 0~4번 옷 이름
         'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']          # 5~9번 옷 이름 (names[정답번호] 로 꺼냄)

In [ ]:
# 옷 사진 20장 보기
for i in range(20):                                     # 0번부터 19번 사진까지
    plt.subplot(4, 5, i + 1)                            # 4행 5열 중 i+1 번째 칸
    plt.imshow(X_train[i], cmap='gray')                 # i번째 사진을 흑백으로 그리기
    plt.title(names[y_train[i]], fontsize=8)            # 정답 번호를 옷 이름으로 바꿔서 제목으로
    plt.axis('off')                                     # 눈금 숨기기
plt.show()                                              # 보여주기

## 1-1. 필터 체험: 필터마다 찾는 모양이 다르다
**필터** = 3×3 숫자표. 사진 위를 훑으며 계산해서 선·모서리 같은 모양을 찾습니다.

In [ ]:
# 13차시의 고양이 사진 준비
if not os.path.exists('sample.png'):                                                # sample.png 가 없으면
    cv2.imwrite('sample.png', cv2.cvtColor(data.chelsea(), cv2.COLOR_RGB2BGR))      # 고양이 사진 저장 (cv2 는 BGR 순서라 바꿈)
img = cv2.imread('sample.png', cv2.IMREAD_GRAYSCALE)                                # 흑백으로 읽어서 숫자표로 가져옴

In [ ]:
# 필터 3개 만들기
filter_v = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])      # 세로 경계를 찾는 필터 (왼쪽 음수, 오른쪽 양수)
filter_h = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])      # 가로 경계를 찾는 필터 (위 음수, 아래 양수)
filter_b = np.ones((10, 10)) / 100                             # 흐리게(블러) 만드는 필터 (10x10 칸의 평균)

# 필터를 적용해서 비교
titles = ['original', 'vertical edge', 'horizontal edge', 'blur']                           # 사진마다 붙일 제목
images = [img] + [cv2.filter2D(img, -1, f) for f in [filter_v, filter_h, filter_b]]          # 원본 + 필터 3개를 적용한 사진 (filter2D(사진, -1, 필터))
for i in range(4):                                  # 사진 4장을 하나씩
    plt.subplot(1, 4, i + 1)                        # 1행 4열 중 i+1 번째 칸에 그리기
    plt.imshow(images[i], cmap='gray')             # 흑백으로 그리기
    plt.title(titles[i], fontsize=9)               # 제목 붙이기
    plt.axis('off')                                 # 눈금 숨기기
plt.show()                                          # 화면에 보여주기

#### 🔁 변형 ①: 내 필터 만들기 (대각선 / 블러 / 선명하게)

In [ ]:
my_filter = np.array([[ 0,  1,  2],             # 대각선(/ 방향) 경계를 찾는 필터
                      [-1,  0,  1],              # 가운데 줄
                      [-2, -1,  0]])             # 아래 줄 (왼쪽 아래 음수, 오른쪽 위 양수 -> / 방향 경계)
blur3 = np.ones((3, 3)) / 9                      # 숫자를 모두 1로 하고 9로 나누면 흐리게 만드는 필터
sharp = np.array([[-1, -1, -1],                  # 가운데 5, 나머지 -1 이면 선명하게 만드는 필터
                  [-1,  5, -1],                  # 가운데 칸을 강하게
                  [-1, -1, -1]])                 # 주변 칸은 빼기

titles = ['original', 'my filter', 'blur 3x3', 'sharp']                                     # 제목
images = [img] + [cv2.filter2D(img, -1, f) for f in [my_filter, blur3, sharp]]               # 필터 적용
for i in range(4):                                  # 4장을 하나씩
    plt.subplot(1, 4, i + 1)                        # 1행 4열
    plt.imshow(images[i], cmap='gray')             # 흑백으로 그리기
    plt.title(titles[i], fontsize=9)               # 제목
    plt.axis('off')                                 # 눈금 숨기기
plt.show()                                          # 보여주기

### 잠깐 멈추기 (답)
1. 세로 경계 필터는 수염 옆선·눈 테두리처럼 **세로 방향 선**을, 가로 경계 필터는 눈 위아래·턱선처럼 **가로 방향 선**을 밝게 찾았다. 숫자 배치(왼쪽 음수·오른쪽 양수 / 위 음수·아래 양수)에 따라 찾는 모양이 달라진다.
2. 블러 필터는 주변 10×10 칸의 **평균**을 내서 세부 모양(털, 경계)이 뭉개져 흐려졌다.

**핵심**: CNN의 `Conv2D`는 이런 필터의 숫자를 사람이 정하지 않고 **학습으로 스스로 찾는다** (필터 숫자 = 학습되는 가중치).

## 1-2. 채널 칸 붙이기
`Conv2D`는 이미지가 `(높이, 너비, 채널)` 모양이길 원합니다. 흑백 사진은 채널이 **1**입니다.

In [ ]:
# 이미지에 채널 칸 붙이기: (장수, 28, 28) -> (장수, 28, 28, 1)
X_train = X_train.reshape(-1, 28, 28, 1)    # -1 : 장수(60000)는 알아서 채워 줌, 마지막 1 = 흑백 채널
X_test = X_test.reshape(-1, 28, 28, 1)      # 테스트용도 똑같이
print(X_train.shape)                        # (60000, 28, 28, 1)

#### 🔁 변형 ②: 손글씨 숫자(mnist)로 바꾸기

In [ ]:
from tensorflow.keras.datasets import mnist                     # 손글씨 숫자 데이터
(Xd_train, yd_train), (Xd_test, yd_test) = mnist.load_data()    # 불러오기 (이름을 다르게 해서 옷 데이터를 덮어쓰지 않음)
Xd_train = Xd_train / 255                                       # 0~1 로 만들기
Xd_test = Xd_test / 255                                         # 테스트용도
Xd_train = Xd_train.reshape(-1, 28, 28, 1)                      # CNN 에 넣으려면 채널 칸 붙이기
Xd_test = Xd_test.reshape(-1, 28, 28, 1)                        # 테스트용도
print(Xd_train.shape, yd_train[:10])                            # (60000, 28, 28, 1) [5 0 4 1 9 2 1 3 1 4]

#### 🔁 변형 ③ ⚠: 컬러 사진이면? (채널 3)
컬러는 `(높이, 너비, 3)`이라 **reshape 할 필요 없이** 이미 채널 칸이 있습니다. `Input(shape=(높이, 너비, 3))`로만 바꾸면 됩니다.
```python
X_train = X_train / 255                       # 정규화는 똑같이 255로 나누기
print(X_train.shape)                          # 예) (50000, 32, 32, 3)
# 만약 채널 칸이 없다면: X_train = X_train.reshape(-1, 32, 32, 3)
```

# 2. CNN 모델 만들기
순서: **필터로 훑기(`Conv2D`) → 크기 줄이기(`MaxPooling2D`) → 펼치기(`Flatten`) → 맞히기(`Dense`)**

In [ ]:
# CNN 모델 만들기: 입력 → 훑기 → 줄이기 → 펼치기 → 출력
model = Sequential()                                        # 층을 쌓을 빈 모델 틀
model.add(Input(shape=(28, 28, 1)))                         # 입력 모양 (높이 28, 너비 28, 채널 1)
model.add(Conv2D(32, (3, 3), activation='relu'))            # 3x3 필터 32개로 훑기 -> (26, 26, 32)
model.add(MaxPooling2D((2, 2)))                             # 가로·세로를 절반으로 줄이기 -> (13, 13, 32)
model.add(Flatten())                                        # 한 줄로 펼치기 -> 13 x 13 x 32 = 5408
model.add(Dense(10, activation='softmax'))                  # 출력층: 10종류 각각일 확률
model.summary()                                             # 층마다 출력 모양과 파라미터 수

목표 결과:
```
conv2d (Conv2D)                 (None, 26, 26, 32)    320       <- (3x3x1 + 1) x 32
max_pooling2d (MaxPooling2D)    (None, 13, 13, 32)    0
flatten (Flatten)               (None, 5408)          0         <- 13 x 13 x 32
dense (Dense)                   (None, 10)            54,090    <- (5408 + 1) x 10
Total params: 54,410
```

### 잠깐 멈추기 (답)
1. `MaxPooling2D`를 지나자 가로·세로가 각각 26 → **13**칸으로 줄었다.
2. 13차시 모델은 101,770개, 오늘 CNN은 **54,410개**로 CNN이 절반 정도로 작다.
3. 3×3 필터 32개면 9 × 32 = **288**개, 나머지 32개는 필터마다 하나씩 더하는 값 → 320.

#### 🔁 변형 ④: 필터 수 바꾸기 (16개 / 64개)
필터 수 = `Conv2D`의 첫 번째 숫자 = 출력 모양의 마지막 숫자. 16개 → 27,210 / 64개 → 108,810

In [ ]:
cnn16 = Sequential()                                        # 빈 모델 틀
cnn16.add(Input(shape=(28, 28, 1)))                         # 입력 모양
cnn16.add(Conv2D(16, (3, 3), activation='relu'))            # 3x3 필터 16개 -> (26, 26, 16)
cnn16.add(MaxPooling2D((2, 2)))                             # 절반 -> (13, 13, 16)
cnn16.add(Flatten())                                        # 13 x 13 x 16 = 2704
cnn16.add(Dense(10, activation='softmax'))                  # 출력층
cnn16.summary()                                             # 합계 27,210

#### 🔁 변형 ⑤: 필터 크기 5x5
필터가 커지면 출력이 더 작아짐: 28 - 5 + 1 = **24**. 파라미터 (5×5×1 + 1) × 32 = 832

In [ ]:
cnn5 = Sequential()                                         # 빈 모델 틀
cnn5.add(Input(shape=(28, 28, 1)))                          # 입력 모양
cnn5.add(Conv2D(32, (5, 5), activation='relu'))             # 5x5 필터 32개 -> (24, 24, 32)
cnn5.add(MaxPooling2D((2, 2)))                              # 절반 -> (12, 12, 32)
cnn5.add(Flatten())                                         # 12 x 12 x 32 = 4608
cnn5.add(Dense(10, activation='softmax'))                   # 출력층
cnn5.summary()                                              # 합계 46,922

#### 🔁 변형 ⑥: Conv2D + MaxPooling2D 2층 쌓기 (32 → 64)
두 번째 `Conv2D`는 들어오는 채널이 **32**라서 파라미터가 (3×3×**32** + 1) × 64 = 18,496

In [ ]:
cnn2 = Sequential()                                         # 빈 모델 틀
cnn2.add(Input(shape=(28, 28, 1)))                          # 입력 모양
cnn2.add(Conv2D(32, (3, 3), activation='relu'))             # 1층: 필터 32개 -> (26, 26, 32)
cnn2.add(MaxPooling2D((2, 2)))                              # 절반 -> (13, 13, 32)
cnn2.add(Conv2D(64, (3, 3), activation='relu'))             # 2층: 필터 64개 -> (11, 11, 64)  (13 - 3 + 1 = 11)
cnn2.add(MaxPooling2D((2, 2)))                              # 절반 -> (5, 5, 64)  (11 / 2 = 5.5 -> 버림 5)
cnn2.add(Flatten())                                         # 5 x 5 x 64 = 1600
cnn2.add(Dense(10, activation='softmax'))                   # 출력층
cnn2.summary()                                              # 합계 34,826 (층은 더 많은데 파라미터는 더 적음!)

#### 🔁 변형 ⑦: CNN 뒤에 Dense 은닉층 추가

In [ ]:
cnn_d = Sequential()                                        # 빈 모델 틀
cnn_d.add(Input(shape=(28, 28, 1)))                         # 입력 모양
cnn_d.add(Conv2D(32, (3, 3), activation='relu'))            # 필터 32개 -> (26, 26, 32)
cnn_d.add(MaxPooling2D((2, 2)))                             # 절반 -> (13, 13, 32)
cnn_d.add(Flatten())                                        # 5408
cnn_d.add(Dense(64, activation='relu'))                     # 은닉층: 노드 64개 (5408 + 1) x 64 = 346,176
cnn_d.add(Dense(10, activation='softmax'))                  # 출력층
cnn_d.summary()                                             # 합계 347,146

#### 🔁 변형 ⑧: 풀링 없이 (MaxPooling2D 의 역할 확인)
풀링을 빼면 Flatten 결과가 26 × 26 × 32 = 21,632로 커져서 파라미터가 216,650으로 **4배 정도** 늘어납니다.

In [ ]:
cnn_np = Sequential()                                       # 빈 모델 틀
cnn_np.add(Input(shape=(28, 28, 1)))                        # 입력 모양
cnn_np.add(Conv2D(32, (3, 3), activation='relu'))           # 필터 32개 -> (26, 26, 32)
cnn_np.add(Flatten())                                       # 풀링 없이 바로 펼치기 -> 21632
cnn_np.add(Dense(10, activation='softmax'))                 # 출력층
cnn_np.summary()                                            # 합계 216,650

#### 🔁 변형 ⑨ ⚠: 컬러 사진용 CNN (채널 3)
`Input`의 마지막 숫자만 3으로. 첫 `Conv2D` 파라미터가 (3×3×**3** + 1) × 32 = **896**으로 바뀜.

In [ ]:
cnn_color = Sequential()                                    # 빈 모델 틀
cnn_color.add(Input(shape=(32, 32, 3)))                     # 컬러 사진: (높이 32, 너비 32, 채널 3)
cnn_color.add(Conv2D(32, (3, 3), activation='relu'))        # 필터 32개 -> (30, 30, 32)
cnn_color.add(MaxPooling2D((2, 2)))                         # 절반 -> (15, 15, 32)
cnn_color.add(Flatten())                                    # 15 x 15 x 32 = 7200
cnn_color.add(Dense(10, activation='softmax'))              # 출력층 (분류할 종류 수에 맞추기)
cnn_color.summary()                                         # 합계 72,906

#### 🔁 변형 ⑩: 분류할 종류 수가 다르면?
출력층 `Dense`의 노드 수 = **분류할 종류 수**. 예) 2종류 → `Dense(2, activation='softmax')` (정답 번호 0, 1). 파라미터 11,138

# 3. 학습시키기

In [ ]:
# 학습 방법 정하기
model.compile(optimizer='adam',                             # 모델의 값을 고쳐주는 방법 (adam)
              loss='sparse_categorical_crossentropy',       # 틀린 정도 계산 방법 (정답이 0~9 번호일 때)
              metrics=['accuracy'])                         # 학습하면서 정확도도 기록

In [ ]:
# 학습용 데이터 전체를 3번 반복해서 학습
history = model.fit(X_train, y_train,                       # 학습용 사진, 학습용 정답
                    epochs=3,                               # 전체를 3번 반복 (문제에서 말한 숫자로 바꾸기)
                    validation_data=(X_test, y_test))       # 매 epoch 끝에 테스트용으로 점수 확인 (val_)
# history.history 안에 epoch 마다 'loss', 'accuracy', 'val_loss', 'val_accuracy' 가 기록됨

- `loss` / `accuracy`: **학습용** 데이터 점수
- `val_loss` / `val_accuracy`: **테스트용**(처음 보는) 데이터 점수

#### 🔁 변형 ⑪: epoch 바꾸기 / 변형 모델 학습시키기
모델 이름만 바꾸면 똑같습니다. **compile → fit** 순서는 항상 같음.
```python
cnn2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])   # 학습 방법
cnn2_history = cnn2.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))         # 5번 반복
```

# 4. 학습 곡선

In [ ]:
# loss(틀린 정도) 곡선: 학습용과 테스트용
plt.plot(history.history['loss'], label='train')            # 학습용 loss (x축 = epoch 번호 0, 1, 2)
plt.plot(history.history['val_loss'], label='test')         # 테스트용 loss (이름이 val_ 로 시작)
plt.xlabel('epoch')                                         # x축 이름
plt.ylabel('loss')                                          # y축 이름
plt.legend()                                                # label 로 붙인 이름표 표시
plt.show()                                                  # 보여주기

In [ ]:
# accuracy(정확도) 곡선: 학습용과 테스트용
plt.plot(history.history['accuracy'], label='train')        # 학습용 정확도
plt.plot(history.history['val_accuracy'], label='test')     # 테스트용 정확도
plt.xlabel('epoch')                                         # x축 이름
plt.ylabel('accuracy')                                      # y축 이름
plt.legend()                                                # 이름표 표시
plt.show()                                                  # 보여주기

**곡선 읽는 법**
- loss는 내려가고 accuracy는 올라가면 → 학습이 잘 되고 있음
- 학습용(train) 선이 테스트용(test) 선보다 좋으면 → 이미 본 문제라 더 잘 맞힘 (정상)
- train 은 계속 좋아지는데 test 는 오히려 나빠지면 → **과대적합** (학습용을 외워버림)

# 5. 13차시 모델과 비교
13차시의 Flatten+Dense 모델을 같은 조건(3번 반복)으로 다시 학습시켜 비교합니다.

In [ ]:
# 13차시 모델(Flatten+Dense)을 같은 데이터로 다시 학습
dense_model = Sequential()                                  # 빈 모델 틀
dense_model.add(Input(shape=(28, 28, 1)))                   # 입력층 (채널 칸 붙인 데이터라 (28, 28, 1))
dense_model.add(Flatten())                                  # 한 줄로 펼치기 (784)
dense_model.add(Dense(128, activation='relu'))              # 노드 128개 은닉층
dense_model.add(Dense(10, activation='softmax'))            # 출력층
dense_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])     # 학습 방법 (CNN 과 똑같이)
dense_history = dense_model.fit(X_train, y_train, epochs=3, validation_data=(X_test, y_test))          # 같은 조건(3번)으로 학습

In [ ]:
# 마지막 epoch 의 테스트용 정확도와 파라미터 수 비교
print('CNN:', round(history.history['val_accuracy'][-1], 4))           # [-1] = 마지막 epoch 값, round(값, 4) = 소수 넷째 자리까지
print('Dense:', round(dense_history.history['val_accuracy'][-1], 4))   # Dense 모델의 마지막 테스트 정확도
print('CNN 파라미터:', model.count_params())                            # 54410
print('Dense 파라미터:', dense_model.count_params())                    # 101770

#### 🔁 변형 ⑫: 두 모델의 테스트 정확도 곡선 겹쳐 그리기

In [ ]:
plt.plot(history.history['val_accuracy'], label='CNN')              # CNN 테스트 정확도
plt.plot(dense_history.history['val_accuracy'], label='Dense')      # Dense 테스트 정확도
plt.xlabel('epoch')                                                 # x축 이름
plt.ylabel('val_accuracy')                                          # y축 이름
plt.legend()                                                        # 이름표 표시
plt.show()                                                          # 보여주기

#### 🔁 변형 ⑬: loss 비교 / 학습용 정확도 비교
`[...]` 안의 이름만 바꾸면 됩니다: `'loss'`, `'accuracy'`, `'val_loss'`, `'val_accuracy'`
```python
print('CNN 테스트 loss:', round(history.history['val_loss'][-1], 4))     # 테스트 loss (작을수록 좋음)
print('CNN 학습 정확도:', round(history.history['accuracy'][-1], 4))      # 학습용 정확도
print('CNN 첫 epoch:', round(history.history['val_accuracy'][0], 4))     # [0] = 첫 번째 epoch
```

# 6. AI가 옷을 맞힐 수 있을까?

In [ ]:
# 테스트 사진 10장: 정답과 AI의 답
pred = model.predict(X_test[:10], verbose=0).argmax(axis=1)     # 앞 10장의 확률 중 가장 큰 번호 (verbose=0 : 진행 표시 숨김)
for i in range(10):                                             # 10장을 하나씩
    plt.subplot(2, 5, i + 1)                                    # 2행 5열
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')          # 채널 칸을 빼서 (28, 28) 로 그리기
    plt.title(names[y_test[i]] + ' -> ' + names[pred[i]], fontsize=7)   # '정답 -> AI의 답'
    plt.axis('off')                                             # 눈금 숨기기
plt.show()                                                      # 보여주기
print('맞힌 개수:', (pred == y_test[:10]).sum(), '/ 10')        # 같으면 True(1) -> 더하면 맞힌 개수

In [ ]:
# AI가 틀린 사진 9장 보기
pred_all = model.predict(X_test, verbose=0).argmax(axis=1)      # 테스트 사진 10000장 전부 예측
wrong = np.where(pred_all != y_test)[0]                         # 예측과 정답이 다른 사진의 번호들
print('틀린 사진 수:', len(wrong), '/', len(y_test))             # 틀린 개수 / 전체
for k in range(min(9, len(wrong))):                             # 틀린 사진 중 앞 9장 (9장보다 적으면 그만큼)
    i = wrong[k]                                                # k번째 틀린 사진의 번호
    plt.subplot(3, 3, k + 1)                                    # 3행 3열
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')          # 그리기
    plt.title('answer ' + names[y_test[i]] + ' / AI ' + names[pred_all[i]], fontsize=7)    # 정답 / AI 의 답
    plt.axis('off')                                             # 눈금 숨기기
plt.show()                                                      # 보여주기

In [ ]:
# 내 사진(고양이)을 AI에게 보여주기
filename = 'sample.png'                                         # 내 사진이면 파일 이름 바꾸기
photo = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)              # 흑백으로 읽기
photo = cv2.resize(photo, (28, 28)) / 255                       # 28x28 로 줄이고 0~1 로 (학습 데이터와 똑같이!)
# photo = 1 - photo                                             # 배경이 흰색인 사진이면 앞의 # 지우기 (옷 데이터는 배경이 검정)
prob = model.predict(photo.reshape(1, 28, 28, 1), verbose=0)[0] # (1장, 28, 28, 1) 모양으로 넣고 첫 번째 결과(10개 확률)
print('AI의 답:', names[prob.argmax()], '| 확신:', round(prob.max() * 100), '%')    # 가장 큰 확률의 이름, 그 확률(%)
plt.imshow(photo, cmap='gray')                                  # 사진 보기
plt.axis('off')                                                 # 눈금 숨기기
plt.show()                                                      # 보여주기

#### 🔁 변형 ⑭: 특정 한 장만 예측하기
```python
i = 0                                                                   # 확인할 사진 번호
prob = model.predict(X_test[i].reshape(1, 28, 28, 1), verbose=0)[0]     # 한 장도 (1, 28, 28, 1) 모양으로 넣어야 함
print('정답:', names[y_test[i]], '| AI:', names[prob.argmax()])         # 정답과 AI 의 답
```

# 7. 개념 이해 예상 문제 + 답

**Q1. 필터란 무엇이고, `Conv2D`에서 필터는 어떻게 정해지나요?**
필터는 이미지 위를 훑으며 계산하는 **작은 숫자표(예: 3×3)** 로, 숫자 배치에 따라 세로선·가로선·대각선 같은 서로 다른 모양을 찾는다. 필터 체험에서는 사람이 숫자를 정했지만, `Conv2D`에서는 필터의 숫자를 **학습을 통해 모델이 스스로 찾는다** (필터 숫자 = 학습되는 가중치).

**Q2. `MaxPooling2D((2, 2))`의 역할은?**
2×2 칸마다 **가장 큰 값 하나만 남겨** 가로·세로를 **절반**으로 줄인다. (26, 26, 32) → (13, 13, 32). 채널 수는 그대로, 파라미터는 **0개**. 크기가 줄어 계산량과 뒤쪽 Dense 층의 파라미터가 줄어든다.

**Q3. 채널이란? 왜 `reshape(-1, 28, 28, 1)`을 하나요?**
채널은 한 칸에 들어 있는 색 정보의 수로 흑백 **1**, 컬러(RGB) **3**. `Conv2D`는 `(높이, 너비, 채널)` 모양을 원하는데 `fashion_mnist`는 `(장수, 28, 28)`이라 채널 칸 1을 붙인다. `-1`은 장수(60000)를 알아서 채워 달라는 뜻.

**Q4. `Conv2D(32, (3, 3))`의 출력이 (26, 26, 32)인 이유는?**
3×3 필터가 28칸 위에 놓일 수 있는 위치가 **28 - 3 + 1 = 26**개라서 가로·세로 26, 필터 **32개**가 각각 결과 한 장씩 만들어 마지막이 32.

**Q5. `Conv2D` 파라미터가 320개인 이유는?**
필터 하나에 3 × 3 × 1(채널) = 9개 + 더하는 값 1개 = 10개, 필터 32개 → **10 × 32 = 320**.

**Q6. Flatten의 역할과 출력 크기는?**
여러 겹의 이미지를 **한 줄로 펼쳐** Dense 층에 넣게 한다. 13 × 13 × 32 = **5408**. 파라미터 0개.

**Q7. 출력층이 `Dense(10, activation='softmax')`인 이유는?**
옷 **10종류**라 노드 10개, `softmax`는 출력을 **합이 1인 확률**로 바꾼다. 가장 큰 확률의 번호(`argmax`)가 AI의 답.

**Q8. CNN이 Dense 모델보다 파라미터가 적은데도 잘 맞히는 이유는?**
Dense는 사진을 처음부터 한 줄로 펼쳐 **이웃한 픽셀의 위치 관계(모양)** 를 잃지만, CNN은 필터로 훑으며 **선·모서리 같은 모양을 그대로 찾고**, 같은 필터를 사진 전체에 다시 써서 파라미터가 적다.

**Q9. `predict(...).argmax(axis=1)`은 무엇을 하나요?**
`predict`는 사진마다 10종류 각각의 확률을 내고, `argmax(axis=1)`은 그중 **가장 큰 확률의 번호**를 골라 AI의 답으로 만든다.

**Q10. 내 사진을 AI에게 보여줄 때 해야 하는 처리는?**
학습 데이터와 똑같이 맞춘다: 흑백으로 읽기 → `cv2.resize(photo, (28, 28))` → `/ 255` → `reshape(1, 28, 28, 1)`. 배경이 흰색이면 `1 - photo`로 뒤집는다 (옷 데이터는 배경이 검정).

# 8. 논술형 예상 문제 + 모범 답안
**쓰는 순서: 결과(숫자) → 해석(왜) → 결론**. ⚠ 정확도는 **내 실행 결과 숫자로 바꿔 쓰세요.**

---
### 논술 1. CNN 모델과 13차시 Dense 모델의 결과를 비교하고 해석하시오.
**결과**: 같은 옷 사진으로 3 epoch 학습했을 때 CNN의 테스트 정확도는 약 **0.90**, Dense 모델은 약 **0.87**로 CNN이 더 높았다. 파라미터는 CNN **54,410개**, Dense **101,770개**로 CNN이 절반 정도였다.

**해석**: Dense 모델은 사진을 처음부터 한 줄(784개)로 펼쳐 이웃한 픽셀끼리의 위치 관계, 즉 옷의 모양 정보를 잃는다. CNN은 3×3 필터가 사진을 훑으며 선·모서리 같은 모양을 찾고, 같은 필터를 사진 전체에 다시 써서 파라미터가 적어도 모양을 잘 파악한다. MaxPooling2D로 크기를 절반으로 줄여 뒤쪽 층의 파라미터도 줄었다.

**결론**: 이미지처럼 모양이 중요한 데이터에는 파라미터가 많은 모델보다 모양을 볼 수 있는 구조(CNN)가 더 효과적이다.

---
### 논술 2. 학습 곡선(loss, accuracy)을 보고 학습 과정을 해석하시오.
**결과**: epoch이 지날수록 학습용과 테스트용 loss는 모두 **내려가고**, accuracy는 모두 **올라갔다**. 마지막 epoch에서 학습용 정확도(약 **0.92**)가 테스트용(약 **0.90**)보다 조금 높았다.

**해석**: 모델이 학습을 반복하며 틀린 정도를 줄이는 방향으로 필터와 가중치를 고쳐 나갔기 때문이다. 학습용 점수가 더 높은 것은 학습용 사진은 여러 번 본 문제이고 테스트용은 처음 보는 사진이기 때문이다. 두 선의 차이가 작으므로 외우기만 한 것이 아니라 처음 보는 사진도 잘 맞히는 **일반화**가 되었다.

---
### 논술 3. epoch을 3에서 50으로 늘리면 항상 더 좋아질까? 이유를 쓰시오.
**답**: 항상 좋아지지는 않는다. 학습용 loss는 계속 줄어들지만 어느 순간부터 테스트용 loss는 줄지 않거나 **올라갈 수 있다**. 모델이 학습용 사진의 세세한 부분까지 **외워버리는 과대적합** 때문이다. 테스트용(val_) 선이 더 이상 좋아지지 않는 지점에서 멈추는 것이 좋고, CNN은 계산이 많아 학습 시간도 크게 늘어난다.

---
### 논술 4. AI가 틀린 사진들을 보고 그 이유를 분석하시오.
**결과**: AI는 테스트용 10000장 중 약 **1000장**(약 10%)을 틀렸다. 주로 **Shirt(셔츠)를 T-shirt·Pullover·Coat로**, 또는 그 반대로 헷갈렸다.

**해석**: 셔츠, 티셔츠, 풀오버, 코트는 모두 몸통과 소매가 있는 **비슷한 모양의 윗옷**이라 28×28 흑백 사진에서는 단추·옷깃 같은 작은 차이가 잘 보이지 않는다. 사람이 봐도 헷갈리는 사진이 많았다. 반면 바지(Trouser)나 가방(Bag)처럼 모양이 뚜렷이 다른 옷은 거의 틀리지 않았다.

**결론**: 모양이 비슷한 종류일수록 구별이 어렵다. 더 큰 해상도나 컬러 사진, 더 많은 학습 데이터를 쓰면 실수를 줄일 수 있을 것이다.

---
### 논술 5. 옷 사진만 학습한 AI에게 고양이 사진을 보여주면 어떻게 되는지, 이유를 쓰시오.
**결과**: AI는 고양이 사진을 **옷 10종류 중 하나**(예: Bag, Shirt)로 답했고, 확신이 꽤 높게 나오기도 했다.

**해석**: 이 AI는 옷 10종류만 배웠고 출력층도 10개 중 하나를 고르게 되어 있어서 "고양이"나 "모르겠다"라고 답할 방법이 없다. softmax는 항상 10개 확률의 합을 1로 만들기 때문에 처음 보는 사진이어도 가장 비슷한 옷을 고른다.

**결론**: AI는 **배운 것만 안다**. 학습 데이터 범위를 벗어난 입력에는 틀린 답을 자신 있게 낼 수 있으므로, 어떤 데이터로 학습했는지 알고 그 범위 안에서 써야 한다.

---
### 논술 6. Conv2D 필터 수(16 / 32 / 64)나 층 수를 바꾸면 어떻게 될지 쓰시오.
**답**: 필터 수를 늘리면 찾을 수 있는 모양의 종류가 많아져 정확도가 오를 수 있지만, 출력 채널이 늘어 Flatten 크기와 파라미터가 커진다(16개 27,210 → 32개 54,410 → 64개 108,810). Conv2D + MaxPooling2D를 한 층 더 쌓으면 이미지가 더 작아져(5×5×64 = 1600) 파라미터는 오히려 줄고(34,826개), 앞 층이 찾은 선·모서리를 조합해 소매·옷깃 같은 더 큰 모양을 찾을 수 있어 정확도가 오르는 경우가 많다. 실제 결과는 테스트용 정확도로 확인해야 한다.

---
### 논술 7. MaxPooling2D 를 빼면 어떻게 될지 쓰시오.
**답**: 풀링이 없으면 Conv2D 출력 (26, 26, 32)가 그대로 펼쳐져 Flatten 크기가 5408에서 **21,632**로 4배가 되고, 뒤쪽 Dense 층 파라미터가 늘어 전체가 54,410개에서 **216,650개**로 커진다. 계산이 많아지고 학습할 값이 많아 과대적합 가능성도 커진다. 풀링은 중요한 값(가장 큰 값)만 남기며 크기를 줄여 효율을 높이는 역할을 한다.

---
## 시험 직전 체크리스트
- [ ] `X_train`, `X_test` **둘 다** `/ 255` 했나?
- [ ] CNN이면 `reshape(-1, 28, 28, 1)` 했나? (`Input` 모양과 똑같아야 함)
- [ ] 출력층 노드 수 = **분류할 종류 수** 인가?
- [ ] `compile` → `fit` 순서, `epochs=` 숫자가 문제와 같은가?
- [ ] 곡선/비교에서 테스트용은 **`val_`** 로 시작하는 이름인가?
- [ ] `cv2.resize(img, (너비, 높이))` 순서가 맞나?
- [ ] 논술형은 **결과(숫자) → 해석(왜) → 결론** 순서로, 숫자는 내 실행 결과로 바꿨나?